# Notebook 3: Inferential Statistics & the Central Limit Theorem

**Difficulty:** ⭐⭐⭐ (Intermediate) | **Estimated Time:** 2.5 hours

---

## The Big Question

> *"We only have data from 100 houses. Can we say anything meaningful about ALL houses in the city?"*

This is the central problem of **inferential statistics**: making confident claims about a large population when you only have a small sample. It is also the engine behind most of machine learning — every model you train is a sample-based approximation of the true underlying pattern.

---

### Topics Covered
1. Population vs Sample
2. Sampling Distributions
3. The Central Limit Theorem (CLT)
4. Standard Error
5. Why CLT is the backbone of ML statistics

---

### Libraries Used
```
numpy, pandas, matplotlib, seaborn, scipy.stats
```

## 1. Why Does This Matter for ML?

Every ML workflow involves **sampling**:

| Situation | Population | Sample |
|-----------|------------|--------|
| House price model | All houses ever sold in the city | Your 5,000-row training dataset |
| Train/test split | Your full dataset | The 80% training split |
| Cross-validation fold | Training data | One fold of k-fold CV |
| A/B test for a new feature | All future users | Users in the experiment |

**The CLT tells us:** even if house prices are wildly skewed (most houses are cheap, a few are extremely expensive), the *average* price from a sufficiently large sample will behave in a very predictable, bell-shaped way. This predictability is what makes confidence intervals, hypothesis tests, and model evaluation metrics *trustworthy*.

Without the CLT, we could not answer questions like:
- "Is my model's RMSE on the test set significantly better than a baseline?"
- "Are the coefficients in my linear regression actually meaningful, or just noise?"
- "How confident am I in this prediction interval?"

## 2. The Soup-Tasting Analogy: Population vs Sample

Imagine a chef has made an enormous pot of soup — 10,000 litres. She needs to know if it is salty enough before the dinner service.

- **The pot of soup** = the **population** (all 10,000 houses in the city)
- **Tasting a spoonful** = drawing a **sample** (surveying 100 houses)
- **The saltiness of the whole pot** = the **population parameter** (true mean house price)
- **The saltiness of your spoonful** = the **sample statistic** (your estimate of the mean)

Key insight: you *cannot* taste the whole pot — it would be eaten before dinner. Similarly, you cannot measure every house in a city. You must **infer** from a spoonful.

### Two critical questions
1. How **accurate** is my spoonful? (Is the sample mean close to the population mean?)
2. How **precise** is it? (How much would the estimate change if I took a different spoonful?)

The **Central Limit Theorem** answers both.

In [ ]:
# ── Setup: import all libraries needed for this notebook ──────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

# Set a random seed so results are reproducible every time you run this notebook
np.random.seed(42)

# Use a clean, professional plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')

# Suppress minor warnings to keep output clean
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully.")
print(f"NumPy version  : {np.__version__}")
print(f"Pandas version : {pd.__version__}")

## 3. Building the Population: 10,000 House Prices

Real-world house prices are **right-skewed**: most houses cluster around a moderate price, but a long tail of luxury properties stretches into the millions. A **lognormal distribution** models this beautifully — it is always positive, and its logarithm is normally distributed.

### Plain English
> If you take the *log* of house prices, they look roughly normal. But the raw prices look skewed with a long right tail.

### Why lognormal?
- House prices can't be negative (unlike a true normal)
- Small differences in percentage terms compound (price = base × location_factor × size_factor × ...)
- The product of many independent positive factors gives a lognormal shape

We will create this population and treat it as the **ground truth** — all 10,000 houses in the city. In practice, you would *never* have this — this simulation exists so we can verify our statistical intuitions.

In [ ]:
# ── Create the synthetic house price population ───────────────────────────────

POPULATION_SIZE = 10_000   # total houses in the city (our "ground truth")

# np.random.lognormal(mean, sigma, size)
# mean=12.5 and sigma=0.5 produce prices roughly centered around exp(12.5) ≈ $270K
# The lognormal is always positive and has a realistic right-skewed shape
raw_population = np.random.lognormal(mean=12.5, sigma=0.5, size=POPULATION_SIZE)

# Round to the nearest dollar for realism
population_prices = np.round(raw_population, 0)

# ── Compute TRUE population parameters (we know these because we created the data)
pop_mean = np.mean(population_prices)       # true mean: μ
pop_std  = np.std(population_prices)        # true standard deviation: σ
pop_median = np.median(population_prices)   # median is less affected by the skew

print("=== POPULATION PARAMETERS (Ground Truth) ===")
print(f"  N (population size) : {POPULATION_SIZE:,} houses")
print(f"  Mean price  (μ)     : ${pop_mean:,.0f}")
print(f"  Median price        : ${pop_median:,.0f}")
print(f"  Std dev     (σ)     : ${pop_std:,.0f}")
print(f"  Min price           : ${population_prices.min():,.0f}")
print(f"  Max price           : ${population_prices.max():,.0f}")
print()
print("Note: mean > median → right-skewed distribution (as expected for house prices)")

In [ ]:
# ── Visualise the population distribution ─────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('House Price Population (N = 10,000)', fontsize=15, fontweight='bold')

# ── Left panel: raw prices (skewed) ──
ax = axes[0]
ax.hist(population_prices / 1000, bins=80, color='steelblue', edgecolor='white',
        alpha=0.8, density=True)
ax.axvline(pop_mean / 1000, color='red',    linestyle='--', linewidth=2,
           label=f'Mean = ${pop_mean/1000:.0f}K')
ax.axvline(pop_median / 1000, color='green', linestyle='--', linewidth=2,
           label=f'Median = ${pop_median/1000:.0f}K')
ax.set_title('Raw House Prices (Right-Skewed)', fontsize=12)
ax.set_xlabel('Price ($K)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.legend(fontsize=10)
ax.annotate('Long right tail\n(luxury homes)', xy=(900, 0.0008), fontsize=9,
            color='darkred', style='italic')

# ── Right panel: log of prices (approximately normal) ──
ax2 = axes[1]
log_prices = np.log(population_prices)
ax2.hist(log_prices, bins=60, color='coral', edgecolor='white',
         alpha=0.8, density=True)
# Overlay the theoretical normal curve to confirm the lognormal assumption
x_range = np.linspace(log_prices.min(), log_prices.max(), 300)
ax2.plot(x_range, stats.norm.pdf(x_range, log_prices.mean(), log_prices.std()),
         'k-', linewidth=2, label='Normal curve fit')
ax2.set_title('log(House Prices) — Approximately Normal', fontsize=12)
ax2.set_xlabel('log(Price)', fontsize=11)
ax2.set_ylabel('Density', fontsize=11)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

print("Key takeaway: raw prices are skewed, but log(prices) are approximately normal.")
print("This skewness is WHY the CLT is so important — real data is rarely perfectly normal.")

## 4. What is a Sampling Distribution?

### Plain English First

Suppose you are a real estate analyst. You survey **30 random houses** from the city and compute their mean price. You get \$285K.

Your colleague does the same survey with a *different* random 30 houses. She gets \$291K.

Your boss does it too — \$279K.

Every sample gives a *slightly different* mean. If you repeated this experiment **1000 times** and collected all those means, what would their distribution look like?

That collection of sample means is called the **sampling distribution of the mean**. It is a distribution *of statistics*, not of raw data.

### Why This Matters
- The **width** of the sampling distribution tells you how *variable* your estimates are
- A **narrow** sampling distribution → you can trust your estimate
- A **wide** sampling distribution → you need a bigger sample

### The Surprising Fact (CLT Preview)
Even though the original house price distribution is skewed, the sampling distribution of means becomes **bell-shaped** as the sample size grows. This is the Central Limit Theorem.

In [ ]:
# ── Simulate the sampling distribution ───────────────────────────────────────
# We will:
#   1. Draw 1000 samples of size n from the population
#   2. Compute the mean of each sample
#   3. Store all 1000 means → this IS the sampling distribution

def simulate_sampling_distribution(population, sample_size, num_samples=1000):
    """
    Draw `num_samples` random samples of `sample_size` from `population`.
    Return an array of the sample means (the sampling distribution).
    """
    sample_means = np.zeros(num_samples)   # pre-allocate array for speed

    for i in range(num_samples):
        # np.random.choice draws `sample_size` values at random (without replacement)
        single_sample = np.random.choice(population, size=sample_size, replace=False)
        sample_means[i] = np.mean(single_sample)   # record this sample's mean

    return sample_means

# ── Run the simulation for three different sample sizes ──
NUM_SAMPLES = 1000   # number of repeated surveys

means_n5   = simulate_sampling_distribution(population_prices, sample_size=5)
means_n30  = simulate_sampling_distribution(population_prices, sample_size=30)
means_n100 = simulate_sampling_distribution(population_prices, sample_size=100)

print("Sampling simulation complete.")
print(f"\nFor n=5  : mean of sample means = ${np.mean(means_n5):,.0f}  | std = ${np.std(means_n5):,.0f}")
print(f"For n=30 : mean of sample means = ${np.mean(means_n30):,.0f}  | std = ${np.std(means_n30):,.0f}")
print(f"For n=100: mean of sample means = ${np.mean(means_n100):,.0f}  | std = ${np.std(means_n100):,.0f}")
print(f"\nTrue population mean            = ${pop_mean:,.0f}")
print("\nObservation: all three are close to the true mean, but the spread decreases as n grows.")

## 5. The Central Limit Theorem (CLT)

### Plain English
> No matter what shape the original distribution has — skewed, uniform, bimodal, exponential — if you take large enough random samples and compute their means, the **distribution of those means will be approximately normal**.

### The Magic
- It does NOT matter that house prices are skewed
- As sample size n increases, the sampling distribution of the mean becomes more and more bell-shaped
- By n=30, the approximation is usually very good
- By n=100, it is excellent for almost any real-world distribution

### The Formula

If the population has mean **μ** and standard deviation **σ**, then the sampling distribution of the mean $\bar{X}$ is:

$$\bar{X} \sim N\left(\mu, \frac{\sigma^2}{n}\right) \quad \text{as } n \to \infty$$

In words:
- The **center** of the sampling distribution equals the population mean μ
- The **spread** of the sampling distribution = σ / √n (called the **Standard Error**)

### Standard Error (SE)

$$SE = \frac{\sigma}{\sqrt{n}}$$

- **SE measures how wrong your sample mean typically is**
- Larger n → smaller SE → more precise estimate
- To halve the SE, you need **4×** the sample size (because of the square root)

In [ ]:
# ── Verify the Standard Error formula empirically ────────────────────────────
# The CLT predicts: std of sample means ≈ population_std / sqrt(n)
# We can check this against our simulation results

print("=== Verifying the Standard Error Formula ===")
print(f"Population std (σ) = ${pop_std:,.0f}")
print()

for n, means in [(5, means_n5), (30, means_n30), (100, means_n100)]:
    # Theoretical prediction from the CLT formula
    predicted_se = pop_std / np.sqrt(n)

    # Empirical measurement: actual std of the 1000 sample means
    empirical_se = np.std(means)

    # Percentage difference to see how well the formula holds
    pct_diff = abs(predicted_se - empirical_se) / predicted_se * 100

    print(f"n = {n:>3d} | Predicted SE = ${predicted_se:>7,.0f}"
          f" | Empirical SE = ${empirical_se:>7,.0f}"
          f" | Difference = {pct_diff:.1f}%")

print()
print("The formula is highly accurate — the CLT works!")
print()
print("Practical implication:")
print("  n=5   → SE is large: individual sample means vary widely")
print("  n=30  → SE is moderate: good for many real-world applications")
print("  n=100 → SE is small: quite precise estimates")
print()
print("To halve the SE you must QUADRUPLE the sample size (cost-benefit tradeoff in data collection)")

In [ ]:
# ── The Main CLT Visualisation ────────────────────────────────────────────────
# Top row: original skewed population
# Bottom row: sampling distribution of means at n=5, n=30, n=100
# We overlay a normal curve to show convergence

fig = plt.figure(figsize=(16, 10))
fig.suptitle('Central Limit Theorem: From Skewed Population to Normal Sampling Distribution',
             fontsize=14, fontweight='bold', y=0.98)

# GridSpec: 2 rows, 3 columns
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

sample_sizes    = [5, 30, 100]
sample_means_list = [means_n5, means_n30, means_n100]
colors          = ['#E07B54', '#5B8DB8', '#6BAE75']   # orange, blue, green

for col_idx, (n, sample_means, color) in enumerate(zip(sample_sizes, sample_means_list, colors)):

    # ── TOP ROW: show the original skewed population ──
    ax_top = fig.add_subplot(gs[0, col_idx])
    ax_top.hist(population_prices / 1000, bins=60, color='steelblue',
                edgecolor='white', alpha=0.7, density=True)
    ax_top.axvline(pop_mean / 1000, color='red', linestyle='--', linewidth=1.8)
    ax_top.set_title(f'Population (n={n} samples drawn from here)', fontsize=10)
    ax_top.set_xlabel('Price ($K)', fontsize=9)
    ax_top.set_ylabel('Density', fontsize=9)
    ax_top.text(0.97, 0.95, 'Skewed!', transform=ax_top.transAxes,
                ha='right', va='top', fontsize=9, color='darkred', style='italic')

    # ── BOTTOM ROW: sampling distribution of means ──
    ax_bot = fig.add_subplot(gs[1, col_idx])
    ax_bot.hist(sample_means / 1000, bins=40, color=color,
                edgecolor='white', alpha=0.75, density=True,
                label=f'Sample means (n={n})')

    # Overlay the theoretical normal curve predicted by CLT
    predicted_mean = pop_mean                      # CLT: center = population mean
    predicted_se   = pop_std / np.sqrt(n)          # CLT: spread = σ / √n
    x_curve = np.linspace(sample_means.min(), sample_means.max(), 300)
    normal_curve = stats.norm.pdf(x_curve, predicted_mean, predicted_se)
    ax_bot.plot(x_curve / 1000, normal_curve / 1000,
                'k-', linewidth=2.5, label='CLT normal prediction')

    ax_bot.axvline(pop_mean / 1000, color='red', linestyle='--',
                   linewidth=1.8, label=f'True mean = ${pop_mean/1000:.0f}K')

    se_text = f'SE = ${predicted_se/1000:.1f}K'
    ax_bot.set_title(f'Sampling Distribution (n={n})\n{se_text}', fontsize=10)
    ax_bot.set_xlabel('Sample Mean Price ($K)', fontsize=9)
    ax_bot.set_ylabel('Density', fontsize=9)
    ax_bot.legend(fontsize=7.5, loc='upper right')

    # Label the shape improvement
    labels = {5: 'Still skewed', 30: 'Nearly normal!', 100: 'Very normal!'}
    ax_bot.text(0.03, 0.95, labels[n], transform=ax_bot.transAxes,
                ha='left', va='top', fontsize=9,
                color='darkgreen' if n >= 30 else 'darkred', fontweight='bold')

plt.show()

print("Key observation: as n increases from 5 → 30 → 100:")
print("  1. The sampling distribution becomes more bell-shaped (CLT convergence)")
print("  2. The distribution becomes narrower (smaller SE = more precise)")
print("  3. The black normal curve fits better and better")

## 6. CLT on Other Distributions

The CLT is universal — it works on *any* distribution, not just lognormal house prices. Let's demonstrate it on three very different shapes:

| Distribution | Shape | Real-world analogy |
|---|---|---|
| **Uniform** | Flat rectangle | Random number between \$100K–\$500K with no preference |
| **Exponential** | Steep drop-off | Time between house sales; most sell quickly, some sit forever |
| **Lognormal** | Right-skewed | Our house price population (already seen) |

**No matter which one we start with, the CLT produces a normal sampling distribution by n=30.**

In [ ]:
# ── CLT on three different population distributions ───────────────────────────

N_POP = 50_000   # large population for each experiment
N_SAMPLES = 2000

# Create three very different population shapes
populations = {
    'Uniform\n(flat)':      np.random.uniform(100_000, 500_000, N_POP),
    'Exponential\n(steep drop)': np.random.exponential(scale=200_000, size=N_POP),
    'Lognormal\n(right-skewed)': np.random.lognormal(12.5, 0.5, N_POP)
}

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
fig.suptitle('CLT is Universal: Works on Any Distribution Shape',
             fontsize=14, fontweight='bold')

sample_sizes_demo = [2, 10, 30]
col_titles = [f'Sampling Dist (n={s})' for s in sample_sizes_demo]

for row_idx, (dist_name, pop) in enumerate(populations.items()):
    pop_m = np.mean(pop)
    pop_s = np.std(pop)

    for col_idx, n in enumerate(sample_sizes_demo):
        ax = axes[row_idx, col_idx]

        # Simulate the sampling distribution for this n
        s_means = np.array([
            np.mean(np.random.choice(pop, size=n, replace=False))
            for _ in range(N_SAMPLES)
        ])

        # Plot histogram of sample means
        ax.hist(s_means / 1000, bins=50, density=True,
                color=plt.cm.Set2(row_idx * 3 + col_idx),
                edgecolor='white', alpha=0.8)

        # Overlay the CLT-predicted normal curve
        se = pop_s / np.sqrt(n)
        x_c = np.linspace(s_means.min(), s_means.max(), 200)
        ax.plot(x_c / 1000,
                stats.norm.pdf(x_c, pop_m, se) / 1000,
                'k-', linewidth=2.0)

        # Row label only on the first column
        if col_idx == 0:
            ax.set_ylabel(dist_name, fontsize=10, fontweight='bold')

        ax.set_title(col_titles[col_idx], fontsize=9)
        ax.set_xlabel('Sample Mean ($K)', fontsize=8)

        # Compute normality score using skewness (lower = more normal)
        skew_val = stats.skew(s_means)
        normality = 'Normal!' if abs(skew_val) < 0.3 else f'Skew={skew_val:.2f}'
        ax.text(0.97, 0.95, normality, transform=ax.transAxes,
                ha='right', va='top', fontsize=8,
                color='green' if abs(skew_val) < 0.3 else 'red')

plt.tight_layout()
plt.show()

print("By n=30 (rightmost column), ALL three distributions have nearly normal")
print("sampling distributions — confirming the CLT works universally.")

## 7. Standard Error Deep Dive

### The Formula

$$SE = \frac{\sigma}{\sqrt{n}}$$

- **σ** = population standard deviation (how variable the data is)
- **n** = sample size (how many observations you collected)

### Practical Interpretation

The SE is the "typical error" when you use a sample mean to estimate the population mean.

If SE = \$15,000, it means your sample mean is typically off by about \$15K from the true mean.

### The Square Root Problem

Notice the **√n** in the denominator — not n itself. This means:

| To reduce SE by... | You need to multiply n by... |
|---|---|
| 2× (half the error) | 4× |
| 3× | 9× |
| 10× | 100× |

This **diminishing returns** effect is why data collection budgets have limits — each additional precision improvement costs exponentially more data.

In [ ]:
# ── Visualise how SE shrinks with sample size ─────────────────────────────────

sample_sizes_range = np.arange(5, 501, 5)   # n from 5 to 500

# Theoretical SE curve: σ / √n
se_theoretical = pop_std / np.sqrt(sample_sizes_range)

# Empirical SE: run the simulation for each n (every 20 steps to save time)
empirical_ns   = np.arange(10, 301, 20)
empirical_ses  = []
for n in empirical_ns:
    s_means = simulate_sampling_distribution(population_prices, sample_size=n,
                                             num_samples=500)
    empirical_ses.append(np.std(s_means))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Standard Error vs Sample Size: Diminishing Returns', fontsize=13, fontweight='bold')

# ── Left: SE vs n (full range) ──
ax1.plot(sample_sizes_range, se_theoretical / 1000, 'b-', linewidth=2.5,
         label='Theoretical: σ/√n')
ax1.scatter(empirical_ns, np.array(empirical_ses) / 1000, color='red',
            s=40, zorder=5, label='Empirical simulation')

# Mark key reference points
for n_mark in [30, 100, 200, 400]:
    se_mark = pop_std / np.sqrt(n_mark)
    ax1.axvline(n_mark, color='gray', linestyle=':', alpha=0.6)
    ax1.text(n_mark + 5, se_mark / 1000 + 0.5,
             f'n={n_mark}\n${se_mark/1000:.1f}K', fontsize=8, color='gray')

ax1.set_xlabel('Sample Size (n)', fontsize=11)
ax1.set_ylabel('Standard Error ($K)', fontsize=11)
ax1.set_title('SE Shrinks Rapidly at First, Then Slowly', fontsize=11)
ax1.legend(fontsize=10)

# ── Right: show cost of halving SE ──
ax2.set_title('Cost of Reducing SE: Diminishing Returns', fontsize=11)
ns_demo  = [25, 100, 400]
ses_demo = [pop_std / np.sqrt(n) / 1000 for n in ns_demo]
colors_bar = ['#E07B54', '#5B8DB8', '#6BAE75']
bars = ax2.bar([f'n={n}' for n in ns_demo], ses_demo,
               color=colors_bar, edgecolor='black', width=0.5)
for bar, se in zip(bars, ses_demo):
    ax2.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.3,
             f'${se:.1f}K', ha='center', fontsize=11, fontweight='bold')
ax2.annotate('', xy=(1, ses_demo[1] + 0.5), xytext=(0, ses_demo[0] + 0.5),
             arrowprops=dict(arrowstyle='->', color='red', lw=1.5))
ax2.text(0.5, max(ses_demo) * 0.7, '4× more data\nhalves the SE',
         ha='center', fontsize=9, color='red', style='italic')
ax2.set_ylabel('Standard Error ($K)', fontsize=11)
ax2.set_xlabel('Sample Size', fontsize=11)

plt.tight_layout()
plt.show()

## 8. Why the CLT is "Magic" for Machine Learning

### Reason 1: It justifies t-tests and confidence intervals
The t-test (covered in Notebook 4) compares two group means and asks: "Is this difference real or just sampling noise?" It only works because the CLT guarantees the sampling distribution of means is approximately normal.

### Reason 2: It explains why model evaluation metrics are stable
When you compute your model's RMSE on a test set, you are computing a sample statistic. The CLT tells you that if you repeated the train/test split many times, the distribution of RMSE values would be approximately normal — which means you can build confidence intervals around your evaluation metric.

### Reason 3: It is why the normal distribution appears everywhere
Many natural phenomena are the sum or average of many independent random effects:
- House price = land value + construction cost + neighborhood premium + market conditions + ...
- Model residuals = sum of many uncorrelated noise sources
- The CLT predicts that all of these sums/averages will be approximately normal

### Reason 4: It validates inference from small datasets
In practice you often can't afford millions of data points. The CLT says n=30 is often "good enough" for many inferential tasks — a remarkably small number.

### The Caveat
The CLT requires:
1. **Independent** samples (no autocorrelation)
2. **Identically distributed** samples (same population)
3. **Finite variance** (extreme heavy tails can slow convergence)

When these are violated (e.g., time-series data, highly fat-tailed distributions), you need to be more careful.

In [ ]:
# ── Practical Application: Confidence Interval using CLT ─────────────────────
# You have ONE sample of 100 houses. Can you estimate the population mean?
# The CLT gives us the tools to build a 95% confidence interval.

# Draw a single sample — this represents what a real analyst would have
MY_SAMPLE_SIZE = 100
my_sample = np.random.choice(population_prices, size=MY_SAMPLE_SIZE, replace=False)

# Step 1: Compute the sample mean and sample standard deviation
sample_mean = np.mean(my_sample)
sample_std  = np.std(my_sample, ddof=1)   # ddof=1: Bessel's correction for sample std

# Step 2: Compute the Standard Error using the sample std (since we don't know σ)
se_estimate = sample_std / np.sqrt(MY_SAMPLE_SIZE)

# Step 3: Build a 95% confidence interval using the normal approximation (CLT)
# For 95% CI, we use z=1.96 (the critical value from the standard normal)
# Interpretation: 95% of such intervals will contain the true mean
z_95 = 1.96
ci_lower = sample_mean - z_95 * se_estimate
ci_upper = sample_mean + z_95 * se_estimate

print("=== Confidence Interval for Mean House Price ===")
print(f"  Sample size              : {MY_SAMPLE_SIZE} houses")
print(f"  Sample mean              : ${sample_mean:,.0f}")
print(f"  Sample std               : ${sample_std:,.0f}")
print(f"  Standard Error (SE)      : ${se_estimate:,.0f}")
print(f"  95% Confidence Interval  : (${ci_lower:,.0f}, ${ci_upper:,.0f})")
print(f"  True population mean     : ${pop_mean:,.0f}")
print()

# Check if the true mean falls inside our confidence interval
captured = ci_lower <= pop_mean <= ci_upper
print(f"  Does the CI capture the true mean? {'YES ✓' if captured else 'NO ✗'}")
print()
print("This is the CLT in action: we used ONE sample to make a confident")
print("statement about the whole population, with a quantified uncertainty.")

In [ ]:
# ── Visualise many confidence intervals to understand what 95% means ──────────
# "95% CI" does NOT mean there's a 95% chance the true mean is in THIS interval.
# It means: if we repeat the sampling procedure many times,
# 95% of the resulting intervals will contain the true mean.

NUM_CI_TRIALS = 50   # draw 50 different samples and build a CI for each
ci_results = []

for _ in range(NUM_CI_TRIALS):
    samp = np.random.choice(population_prices, size=MY_SAMPLE_SIZE, replace=False)
    m    = np.mean(samp)
    s    = np.std(samp, ddof=1)
    se   = s / np.sqrt(MY_SAMPLE_SIZE)
    lo   = m - 1.96 * se
    hi   = m + 1.96 * se
    # Does this interval capture the true population mean?
    captured_flag = lo <= pop_mean <= hi
    ci_results.append((m, lo, hi, captured_flag))

# Count how many intervals actually captured the true mean
capture_rate = sum(r[3] for r in ci_results) / NUM_CI_TRIALS * 100

fig, ax = plt.subplots(figsize=(13, 8))

for i, (m, lo, hi, captured_flag) in enumerate(ci_results):
    color = 'steelblue' if captured_flag else 'crimson'
    # Draw the confidence interval as a horizontal error bar
    ax.plot([lo / 1000, hi / 1000], [i, i], '-', color=color, linewidth=1.5, alpha=0.8)
    ax.plot(m / 1000, i, 'o', color=color, markersize=4)

# Draw the true population mean as a vertical reference line
ax.axvline(pop_mean / 1000, color='red', linewidth=2.5, linestyle='--',
           label=f'True mean = ${pop_mean/1000:.0f}K')

# Legend explaining colors
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='steelblue', label='CI captures true mean'),
    Patch(color='crimson',   label='CI misses true mean'),
] + ax.get_legend_handles_labels()[0], fontsize=10)

ax.set_title(
    f'50 Confidence Intervals from 50 Different Samples\n'
    f'Capture rate = {capture_rate:.0f}% (expected ≈ 95%)',
    fontsize=12
)
ax.set_xlabel('Mean House Price ($K)', fontsize=11)
ax.set_ylabel('Trial number', fontsize=11)

plt.tight_layout()
plt.show()

print(f"\nCapture rate: {capture_rate:.0f}% of the 50 CIs contained the true mean.")
print("This should be close to 95% — confirming the CLT's predictions.")

## 9. Summary: Key Takeaways

| Concept | Plain English | Formula |
|---|---|---|
| **Population** | All houses in the city | N (very large) |
| **Sample** | The houses you actually measured | n (manageable) |
| **Population mean** | True average price (usually unknown) | μ |
| **Sample mean** | Your estimate of μ | x̄ |
| **Sampling distribution** | Distribution of x̄ if you took many samples | — |
| **CLT** | Sampling distribution → Normal as n↑ | x̄ ~ N(μ, σ²/n) |
| **Standard Error** | Typical gap between x̄ and μ | SE = σ/√n |
| **95% CI** | Range that captures μ 95% of the time | x̄ ± 1.96·SE |

### Connection to Next Notebook
Armed with the CLT, we can now ask:
> *"Is the difference between house prices in District A and District B statistically significant, or just sampling noise?"*

This is **hypothesis testing** — covered in Notebook 4.

## 10. Self-Check Questions

Test your understanding before moving on.

---

**Q1.** You survey 50 houses and get a mean price of \$320K. Your friend surveys a *different* 50 and gets \$315K. Why are they different, and is that a problem?

<details>
<summary>Click to reveal answer</summary>

They are different because of **sampling variability** — each sample is a random subset of the population, so every sample produces a slightly different mean. This is completely expected and NOT a problem; it is the fundamental nature of sampling. The sampling distribution of the mean quantifies exactly how much variation to expect. If both samples are from the same population, neither estimate is "wrong" — they are both valid but imprecise estimates. The difference between \$320K and \$315K (\$5K) is likely well within one standard error, meaning it is just normal random fluctuation.
</details>

---

**Q2.** If you double your sample size from 100 to 400, by how much does the standard error shrink?

<details>
<summary>Click to reveal answer</summary>

The SE shrinks by a factor of **2** (i.e., it is halved). 

SE(n=100) = σ/√100 = σ/10  
SE(n=400) = σ/√400 = σ/20  

The ratio is 10/20 = 0.5, so SE is halved. Notice that you need to **quadruple** the sample size to halve the standard error — this is the "square root law" that governs all statistical sampling and explains why data collection has diminishing returns.
</details>

---

**Q3.** The CLT says the sampling distribution of means is approximately normal. Does that mean the original data must be normal?

<details>
<summary>Click to reveal answer</summary>

**No** — this is one of the most important (and surprising) facts about the CLT. The original data can have *any* shape: skewed, uniform, bimodal, exponential, or anything else. The CLT says that regardless of the population's shape, the **means** of sufficiently large samples will be approximately normally distributed. This is why t-tests and confidence intervals work on real-world skewed data (like house prices) — you don't need the raw data to be normal, you need the *sample size* to be large enough that the CLT kicks in (generally n ≥ 30 is a common rule of thumb).
</details>

---

**Q4 (Bonus).** A data scientist says: "Our training dataset has 500 houses. The mean sale price is \$295K with SE = \$8K. So the true mean must be between \$287K and \$303K." What is wrong with this statement?

<details>
<summary>Click to reveal answer</summary>

The statement confuses a **confidence interval** with a certainty claim. The correct statement is: "We are 95% confident that the true mean is between \$279K and \$311K (using 1.96×SE)." Or more precisely: "If we repeated this sampling procedure many times, 95% of the resulting intervals would contain the true mean."

Additionally, ±1 SE gives only a ~68% CI (not 95%). For a 95% CI you need ±1.96 SE, which gives (\$295K - \$15.7K, \$295K + \$15.7K) = (\$279K, \$311K).
</details>